# Tutorial 3: Document Processing with LangChain

In this tutorial, we'll explore document processing techniques using LangChain. We'll cover loading and parsing documents, text splitting, building a simple question-answering system, and implementing semantic search.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

load_dotenv()

llm = ChatGroq(model_name='llama-3.1-8b-instant', temperature=0.7)

# Use OllamaEmbeddings if available, otherwise FakeEmbeddings
try:
    from langchain_ollama import OllamaEmbeddings
    embedding_model = OllamaEmbeddings(model='all-minilm', base_url=os.getenv('OLLAMA_EMBEDDING_URL'))
    _ = embedding_model.embed_query('test')
except Exception:
    from langchain_core.embeddings import FakeEmbeddings
    embedding_model = FakeEmbeddings(size=384)
    print('Ollama not available — using FakeEmbeddings for demo')

print("Setup complete.")

## 1. Loading and Parsing Documents

In [2]:
# Load a single document
loader = TextLoader("sample_documents/sample1.txt")
document = loader.load()

print(f"Content of sample1.txt:\n{document[0].page_content[:200]}...\n")

# Load multiple documents from a directory
dir_loader = DirectoryLoader("sample_documents/", glob="*.txt", loader_cls=TextLoader)
documents = dir_loader.load()

print(f"Number of documents loaded: {len(documents)}")
for i, doc in enumerate(documents):
    print(f"Document {i+1} preview: {doc.page_content[:50]}...")

Content of sample1.txt:
# Comprehensive Overview of Artificial Intelligence

## Table of Contents
1. [Introduction to Artificial Intelligence](#introduction-to-artificial-intelligence)
2. [History of AI](#history-of-ai)
3. [...

Number of documents loaded: 1
Document 1 preview: # Comprehensive Overview of Artificial Intelligenc...


In [3]:
from langchain_community.document_loaders import PyPDFLoader

# Load the PDF
loader = PyPDFLoader('sample_documents/sample2.pdf')
documents = loader.load()
print(f'PDF pages loaded: {len(documents)}')
print(f'First page preview:\n{documents[0].page_content[:200]}...')

PDF pages loaded: 26
First page preview:
Quiet-ST aR: Language Models Can T each Themselves to
Think Before Speaking
Eric Zelikman
Stanford University
Georges Harik
Notbad AI Inc
Yijia Shao
Stanford University
V aruna Jayasiri
Notbad AI Inc
...


## 2. Text Splitting and Chunking

In [4]:
# Create a text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
)

# Split the documents
splits = text_splitter.split_documents(documents)

print(f"Number of splits: {len(splits)}")
print(f"First split preview:\n{splits[0].page_content[:200]}...")

Number of splits: 112
First split preview:
Quiet-ST aR: Language Models Can T each Themselves to
Think Before Speaking
Eric Zelikman
Stanford University
Georges Harik
Notbad AI Inc
Yijia Shao
Stanford University
V aruna Jayasiri
Notbad AI Inc
...


In [ ]:
# Create embeddings and an in-memory vector store from the text splits.
# InMemoryVectorStore (langchain_core) needs no extra native dependency — it's a
# plain dict + numpy cosine similarity, which is all this small demo corpus needs.
# For larger corpora where brute-force search stops scaling, FAISS
# (langchain_community.vectorstores, see Tutorial 5/12) adds approximate
# nearest-neighbor indexing.
vectorstore = InMemoryVectorStore.from_documents(splits, embedding_model)
print(f'Vector store created with {len(splits)} vectors')

## 3. Building a Simple Question-Answering System

In [6]:
# Create LCEL retrieval chain (replaces deprecated RetrievalQA)
rag_prompt = ChatPromptTemplate.from_template(
    "Use the following context to answer the question.\n\nContext: {context}\n\nQuestion: {question}\nAnswer: "
)

rag_chain = (
    {"context": vectorstore.as_retriever(search_kwargs={"k": 3}), "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Ask a question
query = "What is the main topic of these documents?"
answer = rag_chain.invoke(query)

print(f"Question: {query}")
print(f"Answer: {answer}\n")

Question: What is the main topic of these documents?
Answer: The main topic of these documents appears to be mathematics, specifically problem-solving and calculations involving quantities such as eggs, prices, and total earnings.



## 4. Implementing Semantic Search

In [ ]:
# Perform a semantic search
query = "Discuss the importance of AI"
search_results = vectorstore.similarity_search(query, k=3)

print(f"Search query: {query}\n")
print("Top 3 relevant chunks:")
for i, doc in enumerate(search_results):
    print(f"Result {i+1}:\n{doc.page_content[:200]}...\n")

# Use the search results to answer a question
question = "What are some advantages of ai models?"
context = "\n".join([doc.page_content for doc in search_results])

prompt = f"Based on the following context, answer the question: {question}\n\nContext: {context}\n\nAnswer:"
answer = llm.invoke(prompt)

print(f"Question: {question}")
print(f"Answer: {answer.content}")

## Conclusion

In this tutorial, we've explored various aspects of document processing with LangChain, including loading and parsing documents, text splitting, building a simple question-answering system, and implementing semantic search. These techniques form the foundation for more advanced document analysis and information retrieval systems.